# Chapter 5 — Validation, limitations, and the contemporary external-generator challenge

Two distinct jobs, deliberately kept apart:

1. **Sections 1-4** validate what has already been run. Everything is computed from the
   saved artefacts; no check below is asserted from a config flag when it can be verified
   from recorded sample IDs or digests instead.
2. **Section 5** designs a *new* study that has **not been run**. It generates nothing,
   calls no API, and trains nothing. It produces a protocol, sample-size options, cost and
   time expectations, provenance risks and implementation steps, plus a machine-readable
   manifest.

**Nothing in this notebook trains a model or generates an image.**

In [ ]:
import csv
import json
import sys
from pathlib import Path

REPO_ROOT = next(
    parent for parent in [Path.cwd(), *Path.cwd().parents] if (parent / "pyproject.toml").exists()
)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import pandas as pd

from src.evaluation import dissertation as D

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 60)
pd.set_option("display.float_format", lambda v: f"{v:,.4f}")

OUTPUT_ROOT = REPO_ROOT / "outputs"
ctx = D.load_context(OUTPUT_ROOT)
print(f"repository root : {REPO_ROOT}")
print(f"reportable runs : {len(ctx.records)}")

## 1. Reproducibility validation

### 1.1 Run provenance

Every run's identity, status, head type and starting checkpoint. The `inclusion` column
records why an excluded run was excluded, which the chapter needs to state rather than
quietly omit.

In [ ]:
inventory = pd.DataFrame(D.experiment_inventory(ctx))
inventory[["run_id", "protocol", "status", "inclusion", "held_out_generator",
           "head_type", "seed", "torch_version", "manifest_sha256"]]

### 1.2 The exact-reproduction check

The VQDM depth ablation refitted four cells that the standalone VQDM recovery run had
already fitted: same starting checkpoint, same subset seed, same nested subsets, same
byte-identical final test set. The head-only cells therefore act as an end-to-end repeat
of the whole data path — subset construction, checkpoint reload, training, evaluation —
rather than a cached value being read twice.

Agreement here is the reproducibility claim. Disagreement would be a finding.

In [ ]:
repro = pd.DataFrame(D.reproduction_checks(ctx))
repro[["held_out_generator", "cell", "metric", "ablation_value", "recovery_value",
       "difference", "exact"]].reset_index(drop=True)

In [ ]:
summary = (
    repro.groupby("held_out_generator")
    .agg(comparisons=("exact", "size"),
         exact=("exact", "sum"),
         agrees_to_4dp=("agrees_to_4dp", "sum"),
         largest_absolute_difference=("difference", lambda s: s.abs().max()))
    .reset_index()
)
summary

### 1.3 Checkpoint provenance and head separation

Held-out generator alone does **not** identify a run in this project: VQDM is held out
under both a linear and a cosine head. The in-distribution ceiling each recovery curve is
measured against is therefore resolved from the `starting_checkpoint` the adaptation cell
recorded, which names exactly one unseen run.

The check below confirms each ablation's checkpoint resolves to a discovered unseen run
with the *matching* held-out generator, and reports that run's head type.

In [ ]:
provenance = pd.DataFrame(D.provenance_checks(ctx))
provenance[["run_id", "held_out_generator", "starting_checkpoint_owner_run",
            "owner_run_discovered", "owner_run_head_type", "owner_run_held_out",
            "compatibility_warnings"]]

In [ ]:
# Confirm the resolution actually used checkpoint provenance rather than the weaker
# generator-name fallback, across every recovery row the aggregation layer produced.
recovery_rows = pd.DataFrame(ctx.recovery)
match_counts = (
    recovery_rows.groupby(
        ["held_out_generator", "head_type", "in_distribution_reference_match"], dropna=False
    )
    .size()
    .rename("rows")
    .reset_index()
)
match_counts

In [ ]:
resolved = recovery_rows[recovery_rows["in_distribution_reference_match"].notna()]
fallback = resolved[resolved["in_distribution_reference_match"] == "held_out_generator"]
print(f"rows with a resolved in-distribution ceiling : {len(resolved)}")
print(f"resolved via starting_checkpoint provenance  : "
      f"{(resolved['in_distribution_reference_match'] == 'starting_checkpoint').sum()}")
print(f"resolved via the weaker generator fallback   : {len(fallback)}")
print()
print("reference run used, per generator and head:")
print(resolved.groupby(["held_out_generator", "head_type"])
      ["in_distribution_reference_run_id"].unique().to_string())

**Reading.** No row falls back to generator matching. The VQDM reference resolves to the
*linear* unseen run, which is the arm the recovery curves actually started from — the
error this mechanism exists to prevent.

Rows with no resolved ceiling are correct rather than missing: threshold-dependent
metrics at non-default operating points are deliberately left undefined, and
accuracy/precision/recall have no entry in the saved `generalisation_gap` block, so there
is no prevalence-matched ceiling to quote and none is invented.

### 1.4 Manifest and split checks

Digests recorded by the runs themselves, plus the fixed-test-set composition.

In [ ]:
provenance[["held_out_generator", "final_test_sha256", "final_test_size",
            "final_test_prevalence", "final_test_policy", "real_pool_sha256",
            "budget_policy", "budget_policy_violations", "manifest_sha256"]]

In [ ]:
# Are the two ablations measured on the same test set as each other? They should NOT be:
# each generator has its own held-out test set.
for _, row in provenance.iterrows():
    print(f"{row['held_out_generator']:8s} final_test_sha256 {row['final_test_sha256']}")
print()
print("distinct final test sets:", provenance["final_test_sha256"].nunique(),
      "for", len(provenance), "ablations (expected: one per generator)")
print("shared authentic pool  :", provenance["real_pool_sha256"].nunique(),
      "distinct real_pool_sha256 (expected: 1, the same fixed authentic pool)")

## 2. Leakage validation

### 2.1 Adaptation-to-test sample-ID overlap

The check that matters most. Any non-zero intersection would mean a recovery number was
measured on images the cell had been fitted on. Computed by intersecting the saved
adaptation sample IDs against the saved final-test sample IDs, not inferred from the
splitting code.

In [ ]:
overlap = pd.DataFrame(D.adaptation_test_overlap(ctx))
overlap

In [ ]:
print(f"total overlapping sample IDs across all budgets and runs: "
      f"{overlap['overlapping_ids'].sum()}")
print(f"combinations checked: {len(overlap)}, all clean: {overlap['clean'].all()}")

### 2.2 Nested adaptation subsets

Nesting is what makes the recovery curve a curve rather than five unrelated fits. Verified
from the saved sample IDs: each larger budget must be a strict superset of the smaller.
`train_validation_overlap` must be zero, or a cell would be validating on its own
training images.

In [ ]:
nested = pd.DataFrame(D.nested_subset_checks(ctx))
nested[["run_id", "held_out_generator", "subset_seed", "budget", "unique_sample_ids",
        "declared_consumed", "ids_match_declared_count", "train_validation_overlap",
        "previous_budget", "is_superset_of_previous"]]

In [ ]:
checked = nested[nested["is_superset_of_previous"].notna()]
print(f"nesting checks: {len(checked)}, all supersets: {bool(checked['is_superset_of_previous'].all())}")
print(f"ID counts match declared consumption: {bool(nested['ids_match_declared_count'].all())}")
print(f"train/validation overlap total: {nested['train_validation_overlap'].sum()}")

### 2.3 Source-group overlap and content-hash deduplication

These are dataset-construction guarantees, recorded in the manifest audit at build time.

Splitting is **grouped**: `create_grouped_splits` partitions by `source_group`, not by
file, so near-duplicate crops of one source image cannot straddle the train/test boundary.
Authentic images were deduplicated on `sha256_of_file_content` before splitting.

In [ ]:
dataset = D.dataset_provenance(ctx)
for key in ["dataset", "dataset_source", "is_tiny_genimage_subset", "sample_count",
            "split_counts", "generators", "excluded_official_generators",
            "nature_deduplication_key", "deduplicated_repeated_nature_files",
            "deduplicated_across_official_split_boundary", "seed"]:
    print(f"{key:46s} {dataset.get(key)}")

In [ ]:
print("preprocessing policy (why container format cannot predict the class):")
for key, value in (dataset.get("preprocessing") or {}).items():
    print(f"  {key:24s} {value}")

### 2.4 Fixed test-set protection and restart-from-baseline

Two protocol guarantees, both recorded per run:

- The unseen test set is built once per generator and is byte-identical across every
  budget and depth (`final_test_sha256` above).
- Every adaptation cell reloads the *original* starting checkpoint rather than continuing
  from the previous budget's weights (`reload_starting_checkpoint_each_run`), so the
  budgets are independent fits from one common origin rather than a single training run
  sampled at five points.

In [ ]:
import yaml

for record in ctx.ablation_runs():
    resolved = yaml.safe_load((OUTPUT_ROOT / record.run_id / "resolved_config.yaml").read_text())
    fine_tuning = resolved.get("fine_tuning") or {}
    print(f"{ctx.held_out_of(record.run_id):8s} "
          f"reload_starting_checkpoint_each_run={fine_tuning.get('reload_starting_checkpoint_each_run')} "
          f"nested_subsets={fine_tuning.get('nested_subsets')} "
          f"final_test_split={fine_tuning.get('final_test_split_name')}")

### 2.5 Consolidated validation table

Every check above as one pass/fail table. This is exported as `validation_summary.csv`.

In [ ]:
validation = pd.DataFrame(D.validation_summary(ctx))
by_category = (
    validation.groupby("category")
    .agg(checks=("passed", "size"), passed=("passed", "sum"))
    .reset_index()
)
by_category

In [ ]:
failures = validation[~validation["passed"]]
if failures.empty:
    print("All validation checks passed.")
else:
    print(f"{len(failures)} FAILING CHECKS:")
    display(failures)

In [ ]:
validation

## 3. Experimental limitations

Computed from the runs rather than recited from a template: seed counts, test-set size,
saturation counts, generator coverage and excluded runs are all read from what was
actually executed.

In [ ]:
limitations = pd.DataFrame(D.limitations(ctx))
for _, row in limitations.iterrows():
    print("=" * 78)
    print(row["limitation"])
    print("=" * 78)
    print(f"  evidence   : {row['evidence']}")
    print(f"  consequence: {row['consequence']}")
    print()

### 3.1 The internal / external boundary

This is the limitation the external challenge in section 5 exists to address, and it is
worth stating precisely.

Everything measured in Chapter 4 is generalisation to a generator **held out of one
benchmark, assembled at one time, from one source**. That is a real and controlled
measurement of cross-generator generalisation. It is *not* a measurement of how the
detector behaves against generators that did not exist when the benchmark was built.

No image outside the Tiny GenImage subset has been evaluated at any point in this project.

In [ ]:
generators = sorted(
    {str(r["held_out_generator"]) for r in D.recovery_table(ctx) if r["held_out_generator"]}
)
all_known = sorted(set(dataset.get("generators") or []))
print(f"generators in the benchmark        : {all_known}")
print(f"generators held out with a full    : {generators}")
print(f"  depth ablation")
print(f"generators evaluated from outside  : [] (none)")
print(f"  the benchmark")

## 4. Sensitivity and robustness checks already present

Reusing saved data only. No repeated run is invented, and where a robustness question
cannot be answered from what exists, that is stated rather than approximated.

In [ ]:
# What robustness evidence DOES exist, and what does not.
checks = [
    ("Repeat of the same cell in an independent run",
     "YES", f"{int(repro['exact'].sum())}/{len(repro)} metric comparisons exact "
            "(ablation head-only vs standalone recovery run)"),
    ("Multiple subset seeds",
     "NO", "the completed ablations ran subset seed 42 only; an earlier smoke run used "
           "42 and 123 but is excluded as synthetic"),
    ("Multiple training seeds",
     "NO", "training seed 42 only"),
    ("Two classifier heads on the same held-out generator",
     "PARTIAL", "linear and cosine both run for VQDM at 0% adaptation only; no depth "
                "ablation exists for the cosine head"),
    ("Two held-out generators under an identical protocol",
     "YES", "biggan and vqdm, same depths, budgets, subsets and test-set policy"),
    ("Multiple operating points per cell",
     "YES", "default 0.5, adaptation-selected, baseline-unchanged saved for every cell"),
    ("Sensitivity to per-depth learning rate",
     "NO", "rates were probed once on biggan at the 5% budget and carried over to vqdm "
           "unchanged; not re-probed per generator"),
]
pd.DataFrame(checks, columns=["robustness question", "available", "evidence"])

In [ ]:
# The one genuine repeated measurement in the project, quantified.
print("Largest absolute disagreement between the two independent fits of the same cell:")
print(f"  {repro['difference'].abs().max():.10f}")
print()
print("This is an exact-agreement result, not a variance estimate: it shows the pipeline")
print("is deterministic given identical inputs. It says nothing about how much a metric")
print("would move under a different subset seed or training seed, which was never run.")

## 5. Contemporary external-generator challenge

**Status: PROPOSED. Nothing below has been executed.**

No image has been generated, no API has been called, no credential is configured in this
environment, and no external evaluation has been run.

### 5.1 What the generation route actually is

The brief referred to "GPT-6 Astra". Before designing anything around it, the actual
generation route was checked against OpenAI's published documentation, because whether
Astra is an image generator determines what this experiment is even allowed to claim.

**What the documentation says:**

| Fact | Source |
|---|---|
| Model id `gpt-6-astra`, released 2026-09-03 | OpenAI model page |
| Input modalities: text, image. **Output modality: text** | OpenAI API model reference |
| Image generation is a hosted **tool** (`image_generation`) the model can call, listed alongside computer use, code interpreter, web search | OpenAI API model reference |
| The tool selects the image model itself from the GPT Image family (`gpt-image-2`, `gpt-image-1.5`, `gpt-image-1`, `gpt-image-1-mini`) | OpenAI image-generation guide |
| The caller **cannot pin** which GPT Image model is used, and the tool-call result **does not report** which was used | OpenAI image-generation guide |
| The direct Images API **does** accept an explicit model id | OpenAI image-generation guide |

**Conclusion.** GPT-6 Astra is not an image-generation architecture. Calling the study
"the Astra generator" would be a provenance error, not a naming preference. An
Astra-routed image cannot be attributed to any named generator at all.

In [ ]:
# The environment as it actually stands, so the notebook does not imply readiness it lacks.
import importlib.util
import os

print("openai SDK installed     :", importlib.util.find_spec("openai") is not None)
print("OPENAI_API_KEY configured:", bool(os.environ.get("OPENAI_API_KEY")))
print()
print("Neither is present. Generation is not possible from this environment as it stands,")
print("which is consistent with the instruction not to generate anything yet.")

### 5.2 Two defensible designs, answering different questions

| | **Route A (recommended)** | **Route B** |
|---|---|---|
| Call | Images API, pinned model id | Astra + `image_generation` tool |
| Question answered | does the detector generalise to a *named* contemporary generator? | does it generalise to what a contemporary *assistant* produces? |
| Generator identity | recorded exactly | unidentifiable |
| Dissertation label | e.g. "gpt-image-2 challenge" | "Contemporary OpenAI/Astra-mediated image-generation challenge" |
| Main risk | none material | no architectural claim is possible; silent model rotation between batches |

**Recommendation: Route A.** It answers the question the dissertation actually asks —
whether a detector trained on a 2023-era benchmark holds up against a generator released
after it — and it does so with attributable provenance.

If Route B is run anyway for its framing, then the underlying generator must be recorded
as *unidentified*, the study must carry the Astra-mediated name, and no claim about any
generator architecture may be attached to the result.

### 5.3 Protocol

Ordered so that no external image can influence development, training or model selection.

1. **Freeze first.** Nominate the checkpoint and record its sha256 *before* any external
   image exists. The frozen detector is evaluated first and once.
2. **Pre-register the prompt set.** Derive prompts from the ImageNet class vocabulary the
   internal benchmark already uses, so subject matter is not confounded with generator
   identity. Save prompts verbatim with stable ids.
3. **Generate with provenance.** Record every field listed in section 5.6 including the
   raw API response metadata. Hash every image on receipt.
4. **Select authentic comparators by protocol, not by eye.** Draw from the held-out
   authentic pool, class-balanced, matched on resolution, format, compression and colour
   mode, with a recorded seed.
5. **Normalise identically.** Push external images through the existing preprocessing
   policy so container format cannot predict the class — the same guarantee
   `scripts/verify_corrections.py` already enforces internally.
6. **Evaluate once, report separately.** No threshold re-selection, no adaptation, no
   merging into Chapter 4 tables.
7. **Only then, optionally,** run the same limited-data recovery protocol against the
   external set as a second, clearly-separated study.

In [ ]:
# The candidate frozen checkpoints, from the saved runs.
candidates = []
for record in ctx.by_type("unseen_generator"):
    digest = (record.artefact_digests or {}).get("best_checkpoint.pt")
    candidates.append({
        "run_id": record.run_id,
        "held_out_generator": ctx.held_out_of(record.run_id),
        "head_type": record.head_type,
        "best_checkpoint_sha256": digest,
    })
pd.DataFrame(candidates)

Any of these can be the frozen detector. The natural choice is the **linear VQDM** run,
because it is the checkpoint the entire depth ablation started from, so an external result
is directly comparable to the internal recovery curves. That choice must be recorded in
the config before generation, not selected afterwards.

### 5.4 Sample size, cost and time

| Option | Images/class | Total | Purpose |
|---|---|---|---|
| Pilot | 50 | 100 | provenance smoke test; confirm metadata capture works end to end |
| Minimum reportable | 100 | 200 | directional result only |
| **Recommended** | **250** | **500** | matches the internal unseen-test size exactly, so ROC-AUC resolution is directly comparable |

**Cost cannot be stated from this environment.** There is no credential here and the
per-image price of the pinned image model has not been verified. It must be read from
current pricing immediately before the run and recorded in the manifest. Generating 250
images is a small job; the binding constraint is careful provenance capture, not compute
or spend.

**Time.** Generation is minutes to low hours depending on rate limits. Evaluating a frozen
detector on 500 images is a few minutes on the same CPU-only machine that ran the internal
experiments — the 500-image internal test sets evaluate in well under a minute per cell.

### 5.5 Provenance risks

| Risk | Severity | Mitigation |
|---|---|---|
| Unidentifiable underlying generator (Route B) | **high** | choose Route A; otherwise record as unidentified and make no architectural claim |
| Silent model rotation between batches | medium | record model id and response id per image; generate in one session; hash everything |
| Provider-side post-processing (watermarks, C2PA, re-encoding) detectable as a *format* artefact rather than a generator artefact | **high** | strip metadata in the existing preprocessing step; report what was stripped; verify format non-predictiveness on the external set |
| Prompt-subject confound | medium | derive prompts from the same class vocabulary as the internal benchmark |
| Refusals and content filtering skewing the prompt distribution | medium | record refusals; do not silently resample |
| Authentic comparators drawn from a different era/source than the generated set | medium | draw from the held-out authentic pool; match resolution, format, compression, colour mode |

### 5.6 Exact implementation steps

1. Add `openai` to `requirements.txt`; configure a credential **outside** the repository.
2. Write `configs/external_challenge_v1.yaml` recording route, model id, size, quality,
   format, prompt-set id, and the frozen checkpoint sha256.
3. Write `scripts/generate_external_challenge.py` — generation and hashing **only**,
   writing images plus one provenance record per image. No evaluation in this script.
4. Write `scripts/build_external_manifest.py` — assemble the dataset manifest in the
   existing schema so the standard evaluator can read it unmodified.
5. Extend `scripts/verify_corrections.py` with an external-set check: class balance,
   format non-predictiveness, and zero overlap with any internal split.
6. Evaluate the frozen detector with the existing evaluator, writing to a **separate** run
   directory and a **separate** export directory.
7. Report in Chapter 5 only, applying the naming decision from section 5.1.

### 5.7 Machine-readable manifest

Written to `outputs/report/dissertation_results/external_challenge_manifest.json` by
`scripts/build_dissertation_results.py`. Provenance fields that are not yet established
are `null` by design: the schema is fixed before any image exists, so the manifest cannot
later be read as claiming something that was never verified.

In [ ]:
from scripts.build_dissertation_results import external_challenge_manifest

manifest = external_challenge_manifest(ctx)
print(json.dumps(manifest, indent=2))

In [ ]:
# Guard: this notebook must never have produced an external result.
assert manifest["status"] == "proposed_not_executed"
for item in manifest["not_yet_done"]:
    print(f"  - {item}")

## 6. What this chapter can and cannot conclude

Machine-derived group D from the shared findings module, reproduced here because Chapter 5
is where these boundaries have to be stated explicitly.

In [ ]:
for item in D.findings(ctx)["D. Claims that CANNOT be supported from these experiments"]:
    print(f"  - {item}")

The two most likely over-readings, stated plainly:

- **"Full fine-tuning at 5% outperforms head-only at 50%."** Directionally true on every
  metric and both generators, but the VQDM ROC-AUC margin is +0.0014 and the PR-AUC margin
  +0.0057 — both below the 0.02 reliability floor, from a single seed. Write it as
  *equivalence at one tenth the labelling cost*. The F1 margin (+0.0789) and the
  missed-detection counts are the forms that survive.
- **"The detector generalises to modern generators."** Nothing in this project supports
  this. Every result is internal to Tiny GenImage. That is precisely the gap section 5 is
  designed to close, and until it is run the claim cannot be made.